In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
payments_bronze_path = f"{BRONZE_PATH}/payments"
payments_silver_path = f"{SILVER_PATH}/payments"

In [0]:
df_payments_bronze = spark.read.format("delta") \
    .load(payments_bronze_path)

In [0]:
display(df_payments_bronze.limit(20))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
windowSpec = Window.partitionBy("payment_id").orderBy(F.col("updated_at").desc())

df_payments_silver = df_payments_bronze \
    .withColumn(
        "rn",
        F.row_number().over(windowSpec)
    ) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [0]:
df_payments_silver.write.format("delta").mode("append").save(payments_silver_path)

In [0]:
from delta.tables import DeltaTable

payments_silver_table = DeltaTable.forPath(
    spark,
    payments_silver_path
)

payments_silver_table.alias("target") \
    .merge(
        df_payments_silver.alias("source"),
        "source.payment_id = target.payment_id"
    ) \
    .whenMatchedUpdate(
        set = {
            "order_id": "source.order_id",
            "payment_date": "source.payment_date",
            "payment_method": "source.payment_method",
            "amount": "source.amount",
            "payment_status": "source.payment_status",
            "updated_at": "source.updated_at"
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
display(
    spark.read.format("delta") \
        .load(payments_silver_path)
)